# VinTelligence Datathon — Round 1: Part 1 Answers

All answers are verified by code execution on the provided dataset.

**Answer key:**

| Question | Answer |
|---|---|
| Q1 | C |
| Q2 | D |
| Q3 | B |
| Q4 | C |
| Q5 | C |
| Q6 | A |
| Q7 | C |
| Q8 | A |
| Q9 | A |
| Q10 | C |

In [20]:
import pandas as pd
import numpy as np

# Load datasets
orders      = pd.read_csv('dataset/orders.csv')
order_items = pd.read_csv('dataset/order_items.csv')
payments    = pd.read_csv('dataset/payments.csv')
products    = pd.read_csv('dataset/products.csv')
customers   = pd.read_csv('dataset/customers.csv')
geography   = pd.read_csv('dataset/geography.csv')
returns     = pd.read_csv('dataset/returns.csv')
web_traffic = pd.read_csv('dataset/web_traffic.csv')
sales       = pd.read_csv('dataset/sales.csv')

orders['order_date'] = pd.to_datetime(orders['order_date'])

/tmp/ipykernel_128239/2774305204.py:6: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('dataset/order_items.csv')


## Q1 — Median inter-order gap (days) for customers with >1 order

**Answer: C — 144 days**

In [21]:
multi = orders.groupby('customer_id').filter(lambda x: len(x) > 1)
multi_sorted = multi.sort_values(['customer_id', 'order_date'])
multi_sorted['prev_date'] = multi_sorted.groupby('customer_id')['order_date'].shift(1)
pairs = multi_sorted.dropna(subset=['prev_date']).copy()
pairs['gap_days'] = (pairs['order_date'] - pairs['prev_date']).dt.days

median_gap = pairs['gap_days'].median()
print(f'Median inter-order gap: {median_gap:.1f} days')
print('Answer: C')

Median inter-order gap: 144.0 days
Answer: C


## Q2 — Product segment with highest average gross margin ratio

**Answer: D — Standard**

In [22]:
products['gm_ratio'] = (products['price'] - products['cogs']) / products['price']
seg_gm = products.groupby('segment')['gm_ratio'].mean().sort_values(ascending=False)
print(seg_gm)
print(f'\nTop segment: {seg_gm.idxmax()}')
print('Answer: D (Standard)')

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gm_ratio, dtype: float64

Top segment: Standard
Answer: D (Standard)


## Q3 — Most frequent return reason for Streetwear products

**Answer: B — wrong_size**

In [23]:
sw_products = products[products['category'] == 'Streetwear']['product_id']
sw_returns = returns[returns['product_id'].isin(sw_products)]
reason_counts = sw_returns['return_reason'].value_counts()
print(reason_counts)
print(f'\nTop reason: {reason_counts.idxmax()}')
print('Answer: B (wrong_size)')

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Top reason: wrong_size
Answer: B (wrong_size)


## Q4 — Traffic source with lowest average bounce_rate

**Answer: C — email_campaign**

In [24]:
bounce_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(bounce_by_source)
print(f'\nLowest bounce_rate source: {bounce_by_source.idxmin()}')
print('Answer: C (email_campaign)')

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Lowest bounce_rate source: email_campaign
Answer: C (email_campaign)


## Q5 — Percentage of order_items with a promotion applied

**Answer: C — approximately 39%**

In [25]:
total_items = len(order_items)
promo_items = order_items['promo_id'].notna().sum()
pct = promo_items / total_items * 100
print(f'Total line items: {total_items}')
print(f'With promo: {promo_items}')
print(f'Percentage: {pct:.1f}%')
print('Answer: C (39%)')

Total line items: 714669
With promo: 276316
Percentage: 38.7%
Answer: C (39%)


## Q6 — Age group with highest average number of orders per customer

**Answer: A — 55+**

In [26]:
cust_orders = orders.groupby('customer_id').size().reset_index(name='order_count')
cust_with_orders = customers.merge(cust_orders, on='customer_id', how='left')
cust_with_orders['order_count'] = cust_with_orders['order_count'].fillna(0)

non_null_age = cust_with_orders[cust_with_orders['age_group'].notna()]
avg_orders_by_age = non_null_age.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print(avg_orders_by_age)
print(f'\nTop age group: {avg_orders_by_age.idxmax()}')
print('Answer: A (55+)')

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64

Top age group: 55+
Answer: A (55+)


## Q7 — Region with highest total revenue in sales_train.csv

**Answer: C — East**

In [27]:
oi = order_items.copy()
oi['line_revenue'] = oi['quantity'] * oi['unit_price'] - oi['discount_amount'].fillna(0)
order_rev = oi.groupby('order_id')['line_revenue'].sum().reset_index()

orders_zip = orders[['order_id', 'zip']].merge(order_rev, on='order_id')
geo_sub = geography[['zip', 'region']].drop_duplicates('zip')
sales_geo = orders_zip.merge(geo_sub, on='zip', how='left')

rev_by_region = sales_geo.groupby('region')['line_revenue'].sum().sort_values(ascending=False)
print(rev_by_region)
print(f'\nTop region: {rev_by_region.idxmax()}')
print('Answer: C (East)')

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: line_revenue, dtype: float64

Top region: East
Answer: C (East)


## Q8 — Most common payment method for cancelled orders

**Answer: A — credit_card**

In [28]:
cancelled = orders[orders['order_status'] == 'cancelled']
pay_counts = cancelled['payment_method'].value_counts()
print(pay_counts)
print(f'\nTop payment method: {pay_counts.idxmax()}')
print('Answer: A (credit_card)')

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

Top payment method: credit_card
Answer: A (credit_card)


## Q9 — Size with highest return rate (returns / order_items for that size)

**Answer: A — S**

In [29]:
oi_size  = order_items.merge(products[['product_id', 'size']], on='product_id')
ret_size = returns.merge(products[['product_id', 'size']], on='product_id')

oi_count  = oi_size.groupby('size').size().rename('oi_count')
ret_count = ret_size.groupby('size').size().rename('ret_count')

return_rate = (ret_count / oi_count).sort_values(ascending=False)
print(return_rate)
print(f'\nHighest return rate size: {return_rate.idxmax()}')
print('Answer: A (S)')

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64

Highest return rate size: S
Answer: A (S)


## Q10 — Installment plan with highest average payment_value

**Answer: C — 6 installments**

In [30]:
avg_pay = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(avg_pay)
print(f'\nTop installment plan: {avg_pay.idxmax()} installments')
print('Answer: C (6 installments)')

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

Top installment plan: 6 installments
Answer: C (6 installments)
